###Run Shared Libraries

In [0]:
%run ../Misc/SharedLibraries

In [0]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimpurchasecategory"

###Read Bronze table

In [0]:
purchaseCategoryDf = spark.table("bronze.purchcategory")
purchaseCategoryDf.printSchema()
display(purchaseCategoryDf)

###Build Dimension/Fact table

In [0]:
dimPurchaseCategoryDf = purchaseCategoryDf.filter(purchaseCategoryDf.RecordId.isNotNull()
    ).select(
       purchaseCategoryDf.CategoryId,
       F.trim(purchaseCategoryDf.CategoryName).alias("CategoryName"),
       F.when(purchaseCategoryDf.LastProcessedChange_DateTime.isNull(), F.lit("1900-01-01")).otherwise(purchaseCategoryDf.LastProcessedChange_DateTime).cast("timestamp").alias("LastProcessedChange_DateTime"),
       F.from_utc_timestamp(purchaseCategoryDf.DataLakeModified_DateTime,'CST').alias("DataLakeModified_DateTime"),
       purchaseCategoryDf.CategoryGroupId,
       purchaseCategoryDf.RecordId.alias("PurchCategoryRecordId")
    ).withColumn("UpdatedDateTime", F.lit(UpdatedDateTime)
    ).withColumn("PurchCategoryHashKey", F.xxhash64("PurchCategoryRecordId")
    )

display(dimPurchaseCategoryDf)

###Final dataframe

In [0]:
df_final = dimPurchaseCategoryDf

## Write to Silver Schema

In [0]:
saveDeltaTableToCatalog(df_final,"silver",Entity)
